# Final Project
## Machine Learning for Neuroscience
### Gaia Negev and Tzlil Tabib

In [2]:
import json
import pandas as pd


In [ ]:
# load data from json
with open('data/emoset_challenge_1000_augmented.json', 'r') as f:
    data = json.load(f)


In [ ]:
# data to pd.DataFrame
df = pd.DataFrame(data)
df_annotations = pd.DataFrame(df['annotations'].tolist())
df = pd.concat([df.drop('annotations', axis=1), df_annotations], axis=1)
df.head()

# remove image_name column because it's duplicated in image_id
df = df.drop('image_name', axis=1)
# make image_id the index
df = df.set_index('image_id')
df.head()


In [ ]:
# # divide data to train and test
# from sklearn.model_selection import train_test_split
# train_df, test_df = train_test_split(df, test_size=0.2, random_state=42)
# print(f'Train size: {len(train_df)}, Test size: {len(test_df)}')

# # save division to csv
# train_df.to_csv('data/train.csv')
# test_df.to_csv('data/test.csv')

In [3]:
# load the data 
train_df = pd.read_csv('data/train.csv', index_col='image_id')
test_df = pd.read_csv('data/test.csv', index_col='image_id')

# EDA

In [ ]:
train_df.info()

In [ ]:
# print how many unique values are in each column
for column in train_df.columns:
    unique_values = train_df[column].nunique()
    print(f'{column}: {unique_values} unique values')

In [ ]:
train_df.describe()


In [ ]:
# null analysis
train_df.isnull().sum()

In [ ]:
# emotion distribution to see if there is a class imbalance
emotion_counts = train_df['emotion'].value_counts()
print(emotion_counts)
emotion_counts.plot(kind='bar')

Brightness and colorfulness have a very low percentage of missing values, so we'll consider removing them depending on the model (the model's ability to work with missing values).

## Facial Expression Analysis:

In [ ]:
import matplotlib.pyplot as plt
import pandas as pd

def analyze_column(df, col_name, target_col='emotion'):
    """
    Performs a distribution analysis, null check, and correlation 
    with a target column for a specific feature.
    """
    print(f"\n{'='*10} Analyzing Column: {col_name} {'='*10}")
    
    # 0. fill 'missing' in null values for better visualization
    df[col_name] = df[col_name].fillna('missing')
    
    # 1. Distribution of the column
    plt.figure(figsize=(8, 4))
    df[col_name].value_counts().plot(kind='bar')
    plt.title(f'{col_name.replace("_", " ").title()} Distribution')
    plt.xlabel(col_name)
    plt.ylabel('Count')
    plt.show()

    # 2. Null analysis
    null_entries = df[df[col_name].isnull()]
    print(f"Number of null {col_name} entries: {len(null_entries)}")

    if len(null_entries) > 0:
        plt.figure(figsize=(6, 6))
        null_entries[target_col].value_counts().plot(kind='pie', autopct='%1.1f%%')
        plt.title(f'{target_col.title()} distribution when {col_name} is Null')
        plt.show()
    else:
        print(f"No null values found in {col_name}.")

    # 3. Correlation / Cross-tabulation with Target
    non_null_df = df[df[col_name].notnull()]
    if not non_null_df.empty:
        correlation = pd.crosstab(non_null_df[col_name], non_null_df[target_col])
        print(f"\nCross-tabulation ({col_name} vs {target_col}):")
        print(correlation)
        
        correlation.plot(kind='bar', stacked=True, figsize=(10, 6))
        plt.title(f'{col_name.title()} vs {target_col.title()}')
        plt.xlabel(col_name)
        plt.ylabel('Count')
        plt.legend(title=target_col, bbox_to_anchor=(1.05, 1), loc='upper left')
        plt.show()

In [ ]:
for column in ['facial_expression',  'human_action', 'scene']:
    analyze_column(train_df, column)

In [ ]:
# # handle object column separately 
# # each object will be a column with binary values indicating presence or absence of that object in the image
# from sklearn.preprocessing import MultiLabelBinarizer
# mlb = MultiLabelBinarizer()
# object_dummies = mlb.fit_transform(train_df['object'].fillna('missing').apply(lambda x: x.split(',') if isinstance(x, str) else ['missing']))
# object_df = pd.DataFrame(object_dummies, columns=mlb.classes_, index=train_df.index)
# train_df = pd.concat([train_df.drop('object', axis=1), object_df], axis=1)

# try binary PCA to reduce dimensionality of many objects (we can also apply to scene)

In [4]:
import ast
from collections import Counter

# 1. Clean and convert to lists
def clean_to_list(val):
    if isinstance(val, list):
        return val
    if not isinstance(val, str) or val == 'missing':
        return []
    
    # If it looks like a Python list string "['a', 'b']", evaluate it
    if val.startswith('[') and val.endswith(']'):
        try:
            return ast.literal_eval(val)
        except:
            return []
    
    # If it's a comma-separated string "Sunglasses,Flower", split it
    return [item.strip() for item in val.split(',')]

# Apply the conversion
train_df['object_list'] = train_df['object'].fillna('missing').apply(clean_to_list)

# 2. Flatten the list and count occurrences
all_objects = []
for lst in train_df['object_list']:
    all_objects.extend(lst)

object_counts = Counter(all_objects)

# Result
print(f"Count for 'Plant': {object_counts['Plant']}")
print(f"Count for 'Flower': {object_counts['Flower']}")

# convert obect_count to dataframe for better visualization
object_counts_df = pd.DataFrame.from_dict(object_counts, orient='index', columns=['count'])
object_counts_df = object_counts_df.sort_values(by='count', ascending=False)
print(object_counts_df)

Count for 'Plant': 111
Count for 'Flower': 18
            count
Plant         111
Tree           83
House          29
Building       29
Wheel          23
...           ...
Sun hat         1
Maple           1
Bust            1
Billboard       1
High heels      1

[123 rows x 1 columns]


In [5]:
 # add each object as a column to the dataframe with binary values indicating presence or absence of that object in the image
for obj in object_counts.keys():
    train_df[f'object_{obj}'] = train_df['object_list'].apply(lambda x: 1 if obj in x else 0)

/tmp/ipykernel_417736/763284335.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[f'object_{obj}'] = train_df['object_list'].apply(lambda x: 1 if obj in x else 0)
/tmp/ipykernel_417736/763284335.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  train_df[f'object_{obj}'] = train_df['object_list'].apply(lambda x: 1 if obj in x else 0)
/tmp/ipykernel_417736/763284335.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  C

In [6]:
# map the correlation between each object and the emotion
object_emotion_correlation = {}
for obj in object_counts.keys():
    correlation = pd.crosstab(train_df[f'object_{obj}'], train_df['emotion'])
    object_emotion_correlation[obj] = correlation
    print(f"\nCorrelation for object '{obj}':")
    print(correlation)

#correlation matrix for all objects and emotions
object_emotion_matrix = pd.DataFrame()
for obj, correlation in object_emotion_correlation.items():
    correlation = correlation.reindex(columns=train_df['emotion'].unique(), fill_value=0)
    object_emotion_matrix[obj] = correlation.loc[1]  # presence of object


Correlation for object 'Tree':
emotion      amusement  anger  awe  contentment  disgust  excitement  fear  \
object_Tree                                                                  
0                   89     94   88           83       92          90    96   
1                   12      6   14           20        4           7     4   

emotion      sadness  
object_Tree           
0                 85  
1                 16  

Correlation for object 'Plant':
emotion       amusement  anger  awe  contentment  disgust  excitement  fear  \
object_Plant                                                                  
0                    97     90   82           79       84          89    92   
1                     4     10   20           24       12           8     8   

emotion       sadness  
object_Plant           
0                  76  
1                  25  

Correlation for object 'Jacket':
emotion        amusement  anger  awe  contentment  disgust  excitement  fear  \
obj

/tmp/ipykernel_417736/3365092063.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  object_emotion_matrix[obj] = correlation.loc[1]  # presence of object
/tmp/ipykernel_417736/3365092063.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  object_emotion_matrix[obj] = correlation.loc[1]  # presence of object
/tmp/ipykernel_417736/3365092063.py:13: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at onc

In [ ]:
#  facial expression distribution
import matplotlib.pyplot as plt
train_df['facial_expression'].value_counts().plot(kind='bar')
plt.title('Facial Expression Distribution')
plt.xlabel('Facial Expression')
plt.ylabel('Count')
plt.show()

In [ ]:
# null facial expression analysis
null_facial_expression = train_df[train_df['facial_expression'].isnull()]
print(f"Number of null facial expression entries: {len(null_facial_expression)}")

# distribution of emotions in null_facial_expression
null_facial_expression['emotion'].value_counts().plot(kind='pie', autopct='%1.1f%%')
plt.title('Emotion of null facial expression entries')
plt.show()

In [ ]:
# for non-null facial expression, compare the facial expression to the emotion and see if there is a correlation
non_null_facial_expression = train_df[train_df['facial_expression'].notnull()]
correlation = pd.crosstab(non_null_facial_expression['facial_expression'], non_null_facial_expression['emotion'])
print(correlation)
correlation.plot(kind='bar', stacked=True)
plt.title('Facial Expression vs Emotion')
plt.xlabel('Facial Expression')
plt.ylabel('Count')
plt.show()

Conclusions about Facial Expression: 
89% of the samples are missing facial expression
We checked the correlation with emotion to see if we can learn anything about it, we can't. 
Looks like there's no null pattern we can manually overcome
Since most of the sample is missing this field, we'll consider to leave this variable out of the analysis.  

# modeling - 
handle descriptions: 
* tfidf
* bag of words

relevant models to model numeric+categorial variables to predict categorial variable, supervised classification

In [ ]:
# after coding the categorial columns, cluster the metadata to see if there are any patterns in the data

from kmodes.kprototypes import KPrototypes
import numpy as np

# Define the indices of the numerical and categorical features

X = train_df.drop('emotion', axis=1).values
cat_cols = ['facial_expression', 'human_action', 'scene']
num_cols = ['brightness', 'colorfulness']
# Perform k-prototype clustering
kp = KPrototypes(n_clusters=4, init='Cao', n_init=5, verbose=1)
clusters = kp.fit_predict(X, categorical=cat_cols)
# Print the resulting clusters
print(clusters)

# Embedding

## PCA